# Construction Safety Monitor — Training Notebook

This notebook trains a YOLOv8 model to detect PPE violations on construction sites.

**Before running:**
- Go to `Runtime → Change runtime type → T4 GPU`
- Run cells top to bottom in order


## Step 1 — Install dependencies

In [ ]:
!pip install ultralytics roboflow --quiet
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Step 2 — Download dataset from Roboflow

In [ ]:
# Sign up at roboflow.com and get your API key from Settings → API
# Replace the values below with your own project details

from roboflow import Roboflow

API_KEY      = 'YOUR_ROBOFLOW_API_KEY'   # <-- paste your key here
WORKSPACE    = 'YOUR_WORKSPACE'          # e.g. 'john-smith-abc123'
PROJECT_NAME = 'YOUR_PROJECT_NAME'       # e.g. 'construction-ppe'
VERSION      = 1

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
dataset = project.version(VERSION).download('yolov8')

print('Dataset downloaded to:', dataset.location)

## Step 3 — Check dataset structure

In [ ]:
import os
from pathlib import Path

DATA_YAML = Path(dataset.location) / 'data.yaml'
print('data.yaml path:', DATA_YAML)

# Count images per split
for split in ['train', 'valid', 'test']:
    img_dir = Path(dataset.location) / split / 'images'
    if img_dir.exists():
        count = len(list(img_dir.glob('*')))
        print(f'  {split}: {count} images')

## Step 4 — Train the model

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 nano — pre-trained on COCO, we fine-tune on our PPE data
model = YOLO('yolov8n.pt')

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,              # GPU
    project='runs',
    name='ppe_v1',
    exist_ok=True,
    plots=True,
)

print('Training complete!')

## Step 5 — Evaluate on validation set

In [ ]:
# Load best weights and evaluate
best_model = YOLO('runs/ppe_v1/weights/best.pt')
metrics = best_model.val(data=str(DATA_YAML))

print(f'mAP@0.5     : {metrics.box.map50:.3f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.3f}')
print(f'Precision   : {metrics.box.mp:.3f}')
print(f'Recall      : {metrics.box.mr:.3f}')

## Step 6 — Visualise training curves

In [ ]:
from IPython.display import Image, display

# Training loss and metric curves
display(Image('runs/ppe_v1/results.png'))

# Confusion matrix
display(Image('runs/ppe_v1/confusion_matrix.png'))

## Step 7 — Run inference on a test image

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

# Point this at any image you want to test
TEST_IMAGE = 'path/to/your/test_image.jpg'

result = best_model(TEST_IMAGE, conf=0.4)[0]
annotated = result.plot()   # returns BGR numpy array with boxes drawn

cv2_imshow(annotated)

## Step 8 — Download model weights

In [ ]:
from google.colab import files

# Downloads best.pt to your local machine
files.download('runs/ppe_v1/weights/best.pt')
print('Saved best.pt — put this in models/weights/ in your repo')